In [1]:
import os
import sys
with open(sys.argv[0]) as f:
    code = f.read()
import uuid
from math import ceil

import torch
from torch import nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T

torch.backends.cudnn.benchmark = True

import matplotlib.pyplot as plt
import numpy as np

In [2]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

device

'cuda:0'

In [3]:
@torch.compile
def zeropower_via_newtonschulz5(G, steps=3, eps=1e-7):
    """
    Newton-Schulz iteration to compute the zeroth power / orthogonalization of G. We opt to use a
    quintic iteration whose coefficients are selected to maximize the slope at zero. For the purpose
    of minimizing steps, it turns out to be empirically effective to keep increasing the slope at
    zero even beyond the point where the iteration no longer converges all the way to one everywhere
    on the interval. This iteration therefore does not produce UV^T but rather something like US'V^T
    where S' is diagonal with S_{ii}' \sim Uniform(0.5, 1.5), which turns out not to hurt model
    performance at all relative to UV^T, where USV^T = G is the SVD.
    """
    assert len(G.shape) == 2
    a, b, c = (3.4445, -4.7750,  2.0315)
    X = G.bfloat16()
    X /= (X.norm() + eps) # ensure top singular value <= 1
    if G.size(0) > G.size(1):
        X = X.T
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * A @ A
        X = a * X + B @ X
    if G.size(0) > G.size(1):
        X = X.T
    return X

<>:9: SyntaxWarning: invalid escape sequence '\s'
<>:9: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipython-input-981182729.py:9: SyntaxWarning: invalid escape sequence '\s'
  where S' is diagonal with S_{ii}' \sim Uniform(0.5, 1.5), which turns out not to hurt model


In [9]:
G =  torch.randn(512, 256, device=device)
init_norm = torch.linalg.svdvals(G).max()

init_norm 

tensor(38.7661, device='cuda:0')

In [10]:
G_ortho = zeropower_via_newtonschulz5(G)
ortho_norm = torch.linalg.svdvals(G_ortho.float()).max()

ortho_norm

tensor(1.2047, device='cuda:0')

In [11]:
torch.linalg.svdvals(G_ortho.float())

tensor([1.2047, 1.2045, 1.2041, 1.2038, 1.2037, 1.2033, 1.2030, 1.2028, 1.2027,
        1.2025, 1.2022, 1.2020, 1.2016, 1.2014, 1.2013, 1.2013, 1.2005, 1.2004,
        1.2002, 1.1999, 1.1994, 1.1994, 1.1983, 1.1981, 1.1979, 1.1971, 1.1966,
        1.1963, 1.1960, 1.1953, 1.1948, 1.1941, 1.1939, 1.1935, 1.1930, 1.1924,
        1.1916, 1.1907, 1.1899, 1.1898, 1.1891, 1.1889, 1.1885, 1.1873, 1.1867,
        1.1852, 1.1844, 1.1837, 1.1811, 1.1807, 1.1797, 1.1791, 1.1778, 1.1775,
        1.1763, 1.1757, 1.1750, 1.1741, 1.1729, 1.1723, 1.1712, 1.1702, 1.1698,
        1.1687, 1.1679, 1.1667, 1.1649, 1.1635, 1.1628, 1.1616, 1.1600, 1.1583,
        1.1560, 1.1557, 1.1551, 1.1533, 1.1509, 1.1494, 1.1481, 1.1469, 1.1448,
        1.1439, 1.1416, 1.1405, 1.1399, 1.1391, 1.1372, 1.1352, 1.1332, 1.1329,
        1.1294, 1.1287, 1.1274, 1.1256, 1.1242, 1.1217, 1.1188, 1.1174, 1.1140,
        1.1138, 1.1116, 1.1110, 1.1081, 1.1047, 1.1044, 1.1042, 1.1027, 1.1008,
        1.0999, 1.0992, 1.0964, 1.0954, 